# Feature Selection via ANN

## 1) Objective
 

In module 11 we used L1 and L2 regularization in order to reduce the complexity of a model and compared the result to what we got from feature selection using p-values. Now, we want to run **R**ecursive **F**eature **E**limination using an ANN as introduced in ```ANN_Keras.ipynb```. We are demonstrating this procedure with the same diabetes dataset as before, in order to be able to compare the results. 

## 2) Calling Libaries

As before, we will use pre-defined custom functions which are stored in a file called ```Auxiliary.py```. 

In [ ]:
import sys  
sys.path.insert(1, '../15 Codes')

from Auxiliary import *

In [ ]:
help('Auxiliary')

Next, we load all the ```Keras/Tensorflow``` libraries needed for building and running the ANN: 

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import backend as K      # we need later for resetting our model 
from tensorflow.keras import layers # layers for building the ANN

<br>

## 3) Loading, Splitting and Scaling the Data

Next, we load the ```Diabetes2.csv``` data set, separate ```X``` and ```Y```, scale the data accordingly and split the set into training and test set as ususal.

In [ ]:
location = FindMyFile('Diabetes2.csv')

In [ ]:
Data = pd.read_csv(location, sep = ';')
Data.head()

In [ ]:
X    = Data.drop(['label'], axis = 1)
Y    = Data['label']

The data set contains almost 800 data points, i.e. even if we split training and test data in, say 80/20, the number of data points is more than an order of magnitude larger than the number of coefficients.

In [ ]:
X_Train, X_Test, Y_Train, Y_Test = train_test_split(X, Y, test_size = 0.20, random_state = 1)

Scaling the data:

In [ ]:
scaler       = StandardScaler()

XS_Train     = scaler.fit_transform(X_Train)
XS_Test      = scaler.transform(X_Test)

In [ ]:
print(len(Y_Train), len(Y_Test))

In [ ]:
Features = list(X_Train.columns)
print(Features)

<br>

## 4) ANN Model

Last time we found that the best model was obtained using ```elastic net``` with $\alpha_1 = 0.085$ and $\alpha_2 = 0.850$ with an **accuracy of 78%**. This model will serve as reference when benchmarking accuarcy and other quality estimators.<br>
First, we prepare the data for our ANN:

In [ ]:
# One-hot encoding of the classes/labels for Keras
Y_train_oh = keras.utils.to_categorical([0 if y == 'healthy' else 1 for y in Y_Train])
Y_test_oh  = keras.utils.to_categorical([0 if y == 'healthy' else 1 for y in Y_Test])

<br>

We are building the same model as in ```ANN_Keras.ipynb```, but this time in a more compact form. That will help us later, when running feature selection

In [ ]:
def My_Keras_Model(input_dim, Nneurons, NClasses, DArate = .8):
    
    model = keras.Sequential([
        layers.Input(shape = (input_dim,)),
        layers.Dense(Nneurons, activation = 'relu'),
        layers.Dropout(DArate), #dropout rate in order to mitigate overfitting
        layers.Dense(NClasses, activation = 'softmax')
    ])

    model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

    return model

<br>

**4.1) The complete model**

First, we run the complete model and evaluate the result.

In [ ]:
Nneurons  = 8 # we provide a number of neurons for the input layer 
NClasses  = int(np.max(Y_train_oh)+1)
Nfeatures = len(Features)

In [ ]:
keras_model = My_Keras_Model(Nfeatures, Nneurons, NClasses, 0.2)

Starting training loop with ```verbose = 0``` (no output):

In [ ]:
Nepochs = 300 #number of iterative training steps aka epochs
history = keras_model.fit(XS_Train, Y_train_oh, epochs = Nepochs, validation_split = 0.2, verbose = 0)

In [ ]:
plt.plot(history.history['loss'], label = 'Training Loss')
plt.plot(history.history['val_loss'], label ='Validation Loss')
plt.title('Model Loss History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

We can clearly see in the loss plot, that training and validation loss diverge. The fact, that the training loss drops, but not the validation loss, strongly suggests overfitting. We know from the previous modules that the features of this dataset highly correlate. Therefore, this result is not a surprise.<br>
Nevertheless, we evaluate the model using the test data set:

In [ ]:
P = keras_model.predict(XS_Test) # run prediction

In [ ]:
Labels    = ['healthy', 'diabetes'] # labels for plotting
LablesNum = list(np.arange(NClasses))

In [ ]:
PredY     = np.argmax(P, axis = 1) # picking predicted class as the most likely one
Y_TestNum = np.array([0 if y == 'healthy' else 1 for y in Y_Test]).astype(int)

In [ ]:
Accuracy = 100*(PredY == Y_TestNum).sum()/len(PredY)
print(f"Accuracy = {Accuracy: .2f}")

In [ ]:
plot_entropy(P, Y_TestNum, Labels, LablesNum)

In [ ]:
plot_confusion(PredY, Y_TestNum, Labels)

As expected, we obtain similar results as in Module 10 and 11. 

**4.2) Minimal Model**

In this section we want to remove each feature step by step, starting with the least significant one based on test accuray. Since we don't calculate p-values, we need to iterate over each feature in order to find the least significant one. There are different ways how to do that, but running this process recursively in a while loop is probably the most intuitive one.  

Before running the feature selection itself, we first define a function that evaluates the model, since we need to do that during each recursion step.

In [ ]:
def Evaluate_Model(X_Train, X_Test, Y_Train, Y_Test, Nneurons, Nclasses, Nepochs):

    # make sure that shapes fit 
    if X_Train.ndim == 1:
        X_Train, X_Test = X_Train.reshape(-1, 1), X_Test.reshape(-1, 1)
    
    #making sure that ANN is reset each time we run a new fit
    K.clear_session()

    model = My_Keras_Model(X_Train.shape[1], Nneurons, Nclasses, .2)

    model.fit(X_Train, Y_Train, epochs = Nepochs, validation_split = 0.2, verbose = 0)

    _ , acc = model.evaluate(X_Test, Y_Test, verbose = 0)
    
    return acc

Now we are ready for the recursion part. As in Module 10 and 11, we want to monitor which features got dropped and how the accuracy changes.

In [ ]:
def ANN_RFE(X_Train, X_Test, Y_Train, Y_Test, feature_names, Nneurons = 8, Nclasses = 2, Nepochs = 250):

    features = feature_names.copy()
    Xtr      = X_Train.copy()
    Xte      = X_Test.copy()

    Lf       = len(features)

    history = []

    while Lf > 1:
        print(f"\nCurrent features ({len(features)}): {features}")

        Acc_scores = np.zeros((Lf,))

        # dropping each feature, one at the time
        for i in range(Lf):
            Xtr_drop, Xte_drop = np.delete(Xtr, i, axis = 1), np.delete(Xte, i, axis = 1)

            acc           = Evaluate_Model(Xtr_drop, Xte_drop, Y_Train, Y_Test, Nneurons, Nclasses, Nepochs)
            Acc_scores[i] = acc


        # feature whose removal gives BEST accuracy → least important
        remove_idx = np.argmax(Acc_scores)

        removed_feature = features[remove_idx]
        best_acc        = Acc_scores[remove_idx]

        print(f"→ Removing: {removed_feature} (acc = {best_acc: .4f})")

        # here list append is ok, b 
        history.append((removed_feature, best_acc))

        # remove feature
        Xtr = np.delete(Xtr, remove_idx, axis=1)
        Xte = np.delete(Xte, remove_idx, axis=1)
        del features[remove_idx]
        Lf  = len(features)

    print(f"\nFinal remaining feature: {features[0]}")
    return features, history

Now, we are finally ready to run the model:

In [ ]:
final_features, history = ANN_RFE(XS_Train, XS_Test, Y_train_oh, Y_test_oh, feature_names = Features)

Let us plot now how the ANN decided to drop the features.

In [ ]:
FeatureDropped = [f for (f, _) in history]
Accur          = [a*100 for (_, a) in history]

In [ ]:
plt.bar(['None'] + FeatureDropped, [Accuracy] + Accur, color = 'k')
plt.title('accuracy drop')
plt.xlabel('feature dropped')
plt.ylabel('accuracy [%]')
plt.xticks(rotation = 90)
plt.ylim([np.min(Accur[:-2])*0.9, np.max(Accur)*1.05])
plt.show()

Intrestingly, the result differs from our findings in Module 10 and 11. That is because the models we are comparing are not exactly the same: the logistic model includes a constant, which does not exist here. Also note, that drop-out influences the accuracy and we hit **81%** the ANN model!. 